# Lagorii Geo API — Test & Monitor Notebook

Use this to test your deployed API, check cache stats, warm the cache, and verify key safety.

**This is for YOUR testing only** — your customers hit the API directly from their browser via the Shopify JS. This notebook never touches production traffic.

## Setup
Run the cell below once to install dependencies.

In [ ]:
!pip install requests gradio python-dotenv

## Configuration

Point this at your **personal/dev** deployment (CORS open). The `INVALIDATE_SECRET` is only needed for warmup/stats/metrics calls.

**SECURITY NOTE:** never paste your `GOOGLE_MAPS_KEY` or `UPSTASH` tokens here. This notebook only ever needs the public API URL and the invalidate secret. The Google key lives only on the Vercel server.

In [ ]:
import requests, time, json

# Use your DEV deployment for testing (CORS = *)
BASE = "https://lagorii-geo-api-dev.vercel.app"
# For local testing instead:  BASE = "http://localhost:3000"

# Only needed for admin endpoints (warmup / stats / metrics)
INVALIDATE_SECRET = "personal-pick-a-long-random-string"

def pretty(r):
    print(f"HTTP {r.status_code}")
    try: print(json.dumps(r.json(), indent=2))
    except: print(r.text)

## 1. Health check

In [ ]:
pretty(requests.get(f"{BASE}/api/health"))

## 2. Look up a coordinate
First call = `google` (or `nominatim`). Second call = `redis` (cached).

In [ ]:
lat, lng = 12.97, 77.59   # Bangalore Koramangala

t = time.time()
r = requests.get(f"{BASE}/api/geo", params={"lat": lat, "lng": lng})
print(f"took {(time.time()-t)*1000:.0f} ms")
pretty(r)

In [ ]:
# Run again — should now be source: "redis" and much faster
t = time.time()
r = requests.get(f"{BASE}/api/geo", params={"lat": lat, "lng": lng})
print(f"took {(time.time()-t)*1000:.0f} ms")
pretty(r)

## 3. Validation tests
These should all behave correctly (400 for bad input).

In [ ]:
cases = [
    ("Valid India",       {"lat": 12.97, "lng": 77.59}),
    ("Valid UAE",         {"lat": 25.20, "lng": 55.27}),
    ("Null island",       {"lat": 0, "lng": 0}),
    ("Out of range lat",  {"lat": 999, "lng": 77.59}),
    ("String lat",        {"lat": "abc", "lng": 77.59}),
    ("Missing params",    {}),
]
for name, params in cases:
    r = requests.get(f"{BASE}/api/geo", params=params)
    print(f"{name:18} → HTTP {r.status_code}  {r.json()}")

## 4. Warm the cache (admin — needs secret)

In [ ]:
r = requests.get(f"{BASE}/api/geo-warmup", headers={"x-secret": INVALIDATE_SECRET})
pretty(r)

## 5. Cache stats & metrics (admin)

In [ ]:
r = requests.get(f"{BASE}/api/geo-invalidate", params={"mode": "stats"}, headers={"x-secret": INVALIDATE_SECRET})
pretty(r)
print("\n--- metrics ---")
r = requests.get(f"{BASE}/api/metrics", headers={"x-secret": INVALIDATE_SECRET})
pretty(r)

## 6. KEY SAFETY CHECK 🔒
Confirms your Google Maps key is NOT exposed to the browser.

- The API response must **never** contain an API key.
- Admin endpoints must reject calls **without** the secret (401).

In [ ]:
print("CHECK 1 — API response contains no key:")
r = requests.get(f"{BASE}/api/geo", params={"lat": 12.97, "lng": 77.59})
body = r.text.lower()
leaked = any(k in body for k in ["aiza", "google_maps_key", "upstash", "token"])
print("   ❌ POSSIBLE LEAK — inspect response!" if leaked else "   ✅ No key found in response")

print("\nCHECK 2 — admin endpoint rejects missing secret:")
r = requests.get(f"{BASE}/api/geo-warmup")
print("   ✅ 401 as expected" if r.status_code == 401 else f"   ❌ Expected 401, got {r.status_code}")

print("\nCHECK 3 — admin endpoint rejects wrong secret:")
r = requests.get(f"{BASE}/api/geo-invalidate", params={"mode": "stats"}, headers={"x-secret": "wrong"})
print("   ✅ 401 as expected" if r.status_code == 401 else f"   ❌ Expected 401, got {r.status_code}")

## 7. Optional — Gradio dashboard
A simple UI to test coords and view stats. For YOUR use only — not part of the live store.

In [ ]:
import gradio as gr

def lookup(lat, lng):
    try:
        t = time.time()
        r = requests.get(f"{BASE}/api/geo", params={"lat": lat, "lng": lng}, timeout=10)
        ms = (time.time() - t) * 1000
        data = r.json()
        src = data.get("source", "?")
        return f"HTTP {r.status_code} · {ms:.0f} ms · source: {src}\n\n" + json.dumps(data, indent=2)
    except Exception as e:
        return f"Error: {e}"

def stats():
    try:
        r = requests.get(f"{BASE}/api/geo-invalidate", params={"mode": "stats"},
                         headers={"x-secret": INVALIDATE_SECRET}, timeout=10)
        return json.dumps(r.json(), indent=2)
    except Exception as e:
        return f"Error: {e}"

def warmup():
    try:
        r = requests.get(f"{BASE}/api/geo-warmup",
                         headers={"x-secret": INVALIDATE_SECRET}, timeout=60)
        return json.dumps(r.json(), indent=2)
    except Exception as e:
        return f"Error: {e}"

PRESETS = {
    "Bangalore (12.97, 77.59)": (12.97, 77.59),
    "Mumbai (19.07, 72.87)":    (19.07, 72.87),
    "Delhi (28.61, 77.23)":     (28.61, 77.23),
    "Dubai (25.20, 55.27)":     (25.20, 55.27),
    "London (51.50, -0.12)":    (51.50, -0.12),
}

with gr.Blocks(title="Lagorii Geo API") as demo:
    gr.Markdown(f"### Lagorii Geo API Tester\nTarget: `{BASE}`")
    with gr.Row():
        lat_in = gr.Number(label="Latitude", value=12.97)
        lng_in = gr.Number(label="Longitude", value=77.59)
    preset = gr.Dropdown(list(PRESETS.keys()), label="Quick preset")
    preset.change(lambda p: PRESETS[p] if p else (12.97, 77.59),
                  inputs=preset, outputs=[lat_in, lng_in])
    out = gr.Textbox(label="Response", lines=12)
    gr.Button("Look up", variant="primary").click(lookup, [lat_in, lng_in], out)
    with gr.Row():
        gr.Button("Cache Stats").click(stats, None, out)
        gr.Button("Warm Cache").click(warmup, None, out)

demo.launch()